In [ ]:
!pip install -q diffusers transformers accelerate safetensors

import os, gc, re, json, zipfile
from pathlib import Path
import torch
from diffusers import (StableDiffusionXLPipeline, DPMSolverMultistepScheduler,
                       AutoencoderKL)
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          CLIPVisionModelWithProjection)
from PIL import Image

# ================= CẤU HÌNH =================
TEST_LIMIT   = 3        # vẽ thử 3 ảnh đầu; đặt None để vẽ hết
FORCE_REDO   = False    # True = vẽ lại cả ảnh đã có
USE_LLM      = True     # True = Qwen viết mô tả cảnh; False = dùng image_prompt gốc
USE_REF      = True     # True = dùng ảnh tham chiếu (IP-Adapter)
REF_SCALE    = 0.55     # 0.4-0.7. Cao = giống ảnh ref hơn nhưng cứng hơn
IMG_W, IMG_H = 832, 1216
BASE_SEED    = 2026

# TỰ VIẾT, đừng để AI viết (tối đa ~20 từ, chỉ ngoại hình + quần áo)
CHARACTER_BIBLE = ("1boy, young Vietnamese man, 21 years old, short black hair, "
                   "thin face, light grey buttoned shirt, dark trousers")

STYLE_PREFIX = ("masterpiece, best quality, very aesthetic, 2D western animation, "
                "cel-shaded, clean thin linework, soft shading, warm cinematic lighting, "
                "muted earthy colors")

NEG_PROMPT = ("lowres, bad anatomy, bad hands, extra fingers, deformed face, "
              "blurry, watermark, text, signature, photorealistic, 3d render, "
              "chibi, big anime eyes, thick black shadows, military uniform, "
              "chinese robe, hat")

# Thay các cụm AI hay viết sai
FIXES = {
    "ancient ho chi minh city": "Saigon 1911",
    "ho chi minh city": "Saigon 1911",
    "past tense narration": "",
}

# ================= ĐƯỜNG DẪN =================
sb_files = list(Path("/kaggle/input").rglob("storyboard.json"))
assert sb_files, "Không thấy storyboard.json trong /kaggle/input"
SB_PATH = sb_files[0]
print("Kịch bản:", SB_PATH)

ref_imgs = []
for d in Path("/kaggle/input").rglob("refs"):
    if d.is_dir():
        ref_imgs += sorted(p for p in d.rglob("*")
                           if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print("Ảnh tham chiếu:", [p.name for p in ref_imgs] or "KHÔNG CÓ")

OUT_DIR = Path("/kaggle/working/kaggle_bundle")
SCENES_DIR = OUT_DIR / "scenes"
SCENES_DIR.mkdir(parents=True, exist_ok=True)

sb = json.loads(SB_PATH.read_text(encoding="utf-8"))
scenes = sb["scenes"][:TEST_LIMIT] if TEST_LIMIT else sb["scenes"]

# ================= HÀM PHỤ =================
def clean(text, max_words):
    text = re.sub(r"\s+", " ", text).strip(" \"'\n\r\t.")
    low = text.lower()
    for k, v in FIXES.items():
        low = low.replace(k, v)
    return " ".join(low.split()[:max_words])

def fit_77(prefix, char, scene, tok):
    """Cắt phần cảnh cho vừa 77 token CLIP, ưu tiên giữ style + nhân vật."""
    words = scene.split()
    while words:
        p = ", ".join([prefix, char, " ".join(words)])
        if len(tok(p).input_ids) <= 77:
            return p
        words.pop()
    return ", ".join([prefix, char])

# ================= BƯỚC 1: QWEN VIẾT MÔ TẢ CẢNH =================
if USE_LLM:
    print("🧠 Nạp Qwen2.5-3B viết mô tả cảnh...")
    mid = "Qwen/Qwen2.5-3B-Instruct"
    tk = AutoTokenizer.from_pretrained(mid)
    llm = AutoModelForCausalLM.from_pretrained(mid, dtype=torch.float16, device_map="auto")
    for sc in scenes:
        msg = [
            {"role": "system", "content":
             "Output ONLY 12 to 18 English words, comma separated: camera angle "
             "(wide shot / medium shot / close-up), what the character is doing, "
             "and the setting. Vietnam in 1911 (Saigon port, ships, colonial streets, "
             "villages). Do NOT describe the character's face or clothes. "
             "NO art style words, NO full sentences."},
            {"role": "user", "content": f"Narration: {sc['speech']}\nIdea: {sc.get('image_prompt','')}"},
        ]
        t = tk.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        inp = tk([t], return_tensors="pt").to(llm.device)
        out = llm.generate(**inp, max_new_tokens=50, do_sample=True, temperature=0.3)
        sc["scene_desc"] = clean(tk.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True), 20)
    del llm, tk
    gc.collect(); torch.cuda.empty_cache()
else:
    for sc in scenes:
        sc["scene_desc"] = clean(sc.get("image_prompt", ""), 20)

# ================= BƯỚC 2: VẼ ẢNH =================
print("🎨 Nạp Animagine XL 3.1...")
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)

use_ip = USE_REF and len(ref_imgs) > 0
kw = {}
if use_ip:
    enc = CLIPVisionModelWithProjection.from_pretrained(
        "h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16)
    kw["image_encoder"] = enc

pipe = StableDiffusionXLPipeline.from_pretrained(
    "cagliostrolab/animagine-xl-3.1", vae=vae,
    torch_dtype=torch.float16, use_safetensors=True, **kw)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True)

if use_ip:
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                         weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors")
    ref_image = Image.open(ref_imgs[0]).convert("RGB")
    print("Dùng ảnh ref:", ref_imgs[0].name)

pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
try:
    pipe.to("cuda")                      # nhanh hơn
except torch.cuda.OutOfMemoryError:
    pipe.enable_model_cpu_offload()      # dự phòng nếu hết VRAM

for sc in scenes:
    sid = sc["id"]
    out_img = SCENES_DIR / f"{sid:03d}.png"
    if out_img.exists() and not FORCE_REDO:
        print(f"⏭️ {out_img.name} đã có, bỏ qua"); continue

    has_char = sc.get("use_character", True)     # thêm "use_character": false vào cảnh không có nhân vật
    prompt = fit_77(STYLE_PREFIX, CHARACTER_BIBLE if has_char else "no humans, scenery",
                    sc["scene_desc"], pipe.tokenizer)
    sc["image_prompt_final"] = prompt
    print(f"🖼️ {out_img.name} | {prompt}")

    args = dict(prompt=prompt, negative_prompt=NEG_PROMPT, width=IMG_W, height=IMG_H,
                num_inference_steps=26, guidance_scale=6.5,
                generator=torch.Generator("cuda").manual_seed(BASE_SEED + sid))
    if use_ip:
        pipe.set_ip_adapter_scale(REF_SCALE if has_char else 0.0)
        args["ip_adapter_image"] = ref_image
    pipe(**args).images[0].save(out_img)

(OUT_DIR / "storyboard_enriched.json").write_text(
    json.dumps(sb, ensure_ascii=False, indent=2), encoding="utf-8")

zip_out = Path("/kaggle/working/kaggle_output_pack.zip")
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.rglob("*"):
        if f.is_file():
            zf.write(f, f.relative_to(OUT_DIR))
print("🎉 Xong:", zip_out)

# Xem nhanh ảnh vừa vẽ
from IPython.display import display
for p in sorted(SCENES_DIR.glob("*.png"))[:3]:
    display(Image.open(p).resize((416, 608)))